# Systematic Review Automation — Google Colab
**LLMs in Qualitative Data Analysis (2023–2026)**

Run cells top to bottom. Cell 5 gives you a public URL — no accounts needed.

---
### Before you start — add your OpenAI API key as a Colab Secret

1. Click the **🔑 key icon** in the left sidebar (or go to **Runtime → Manage secrets**)
2. Click **+ Add new secret**
3. Name: `OPENAI_API_KEY` · Value: your `sk-...` key
4. Toggle **Notebook access** ON

Your key is stored encrypted in your Google account — it is **never** visible in notebook output, never sent to ngrok, and never stored in any repository.


## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/qbzdm2024/Zhihong.github.io.git'
BRANCH   = 'claude/systematic-review-automation-oC3BZ'
REPO_DIR = '/content/sysrev'
APP_DIR  = f'{REPO_DIR}/systematic-review'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo already present — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {APP_DIR}
print('Working directory:', os.getcwd())

In [ ]:
%%capture
!pip install \
    "fastapi==0.115.0" \
    "uvicorn[standard]" \
    "pydantic>=2.7.0" \
    "pydantic-settings>=2.3.0" \
    "openai>=1.50.0" \
    "rispy>=0.9.0" \
    "python-multipart" \
    "python-dotenv" \
    "PyMuPDF"

In [ ]:
import fastapi, openai, pydantic
print(f'fastapi {fastapi.__version__} | openai {openai.__version__} | pydantic {pydantic.__version__}')

# Create data directories
for d in ['data/raw','data/deduped','data/screened','data/extracted','data/output','data/pdfs']:
    os.makedirs(d, exist_ok=True)
print('Data directories ready.')

## Cell 2 — Load API key from Colab Secrets

This reads your key from the encrypted Colab Secrets vault and writes it to a local `.env` file that only exists inside this Colab session. The key is **never printed**.

In [ ]:
from google.colab import userdata

# Read from Colab Secrets (set via the key icon in the left sidebar)
try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    raise RuntimeError(
        'OPENAI_API_KEY not found in Colab Secrets.\n'
        'Go to: Runtime → Manage secrets → Add new secret\n'
        'Name: OPENAI_API_KEY | Value: sk-...'
    )

if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith('sk-'):
    raise ValueError('OPENAI_API_KEY secret does not look valid (must start with sk-)')

# Write to .env — the server reads this on startup
with open('.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')
    f.write('MODEL_TITLE_SCREENING=gpt-4o-mini\n')
    f.write('MODEL_FULLTEXT_SCREENING=gpt-4o\n')
    f.write('MODEL_EXTRACTION=gpt-4o\n')
    f.write('MODEL_QA_ASSESSMENT=gpt-4o\n')
    f.write('MODEL_AGENT2_SCREENING=gpt-4o\n')
    f.write('MODEL_AGENT2_EXTRACTION=gpt-4o-mini\n')
    f.write('CONFIDENCE_THRESHOLD=0.80\n')

# Quick sanity-check (does NOT print the key)
print(f'API key loaded: sk-...{OPENAI_API_KEY[-4:]}')
print('.env written. Server will start pre-configured — no wizard needed.')

## Cell 3 — Upload your search export files

Supported: `.ris` `.csv` `.bib` `.json` `.xml` (PubMed XML)

Export from: PubMed · Scopus · Web of Science · PsycINFO · ACM DL · IEEE Xplore · ERIC

> Skip and come back later if you don't have files ready yet.

In [ ]:
from google.colab import files as colab_files

SUPPORTED = ('.ris', '.csv', '.bib', '.json', '.xml')

print('Select your search export files:')
uploaded = colab_files.upload()

saved, skipped = [], []
for fname, content in uploaded.items():
    if any(fname.lower().endswith(ext) for ext in SUPPORTED):
        dest = f'data/raw/{fname}'
        with open(dest, 'wb') as f:
            f.write(content)
        saved.append(f'{fname} ({len(content):,} bytes)')
    else:
        skipped.append(fname)

if saved:
    print(f'\n  Saved to data/raw/:')
    for s in saved: print(f'    {s}')
if skipped:
    print(f'  Skipped (unsupported format): {skipped}')

!ls -lh data/raw/

## Cell 4 — (Optional) Upload PDFs for full-text screening

Name each file `<record_id>.pdf`. You can also upload them later through the UI after screening.

In [ ]:
from google.colab import files as colab_files

print('Select PDF files (click Cancel to skip):')
try:
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        if fname.lower().endswith('.pdf'):
            dest = f'data/pdfs/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  Saved: {dest}')
        else:
            print(f'  Skipped (not PDF): {fname}')
except Exception:
    print('No PDFs uploaded — you can add them later via the UI.')

## Cell 5 — Start server + open public URL

Uses **Cloudflare Tunnel** — no account, no token, completely free.
A new random HTTPS URL is generated each session.

> The API key is already loaded from Colab Secrets — the setup wizard will be skipped automatically.

In [ ]:
import subprocess, time, re, sys, os, urllib.request
from IPython.display import display, HTML

APP_DIR = '/content/sysrev/systematic-review'
PORT = 8000

# ── Kill any leftover processes ──────────────────────────
os.system(f'fuser -k {PORT}/tcp 2>/dev/null; pkill cloudflared 2>/dev/null; true')
time.sleep(1)

# ── Download cloudflared (if not already present) ────────
if not os.path.exists('/content/cloudflared'):
    print('Downloading cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
         -O /content/cloudflared && chmod +x /content/cloudflared
    print('cloudflared ready.')

# ── Start FastAPI server ─────────────────────────────────
server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.main:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=APP_DIR,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

# Wait until server responds
print('Starting server', end='')
for _ in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/api/pipeline/status', timeout=2)
        print(' ready.')
        break
    except Exception:
        print('.', end='', flush=True)
else:
    print('\nERROR: Server did not start. Re-run this cell.')

# ── Start Cloudflare tunnel ──────────────────────────────
tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Extract the public URL from cloudflared output
public_url = None
print('Opening Cloudflare tunnel', end='')
for _ in range(30):
    line = tunnel.stdout.readline().decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print(' done.')
        break
    print('.', end='', flush=True)

if not public_url:
    print('\nCould not get tunnel URL. Re-run this cell.')
else:
    app_url = f'{public_url}/app'
    api_url = f'{public_url}/docs'

    print(f'\n{"+"*60}')
    print(f'  UI:   {app_url}')
    print(f'  API:  {api_url}')
    print(f'{"+"*60}')
    print('URL is valid for this session only. Save to Drive when done.')

    display(HTML(f'''
    <div style="margin:16px 0;padding:20px;background:#0f1117;
      border:2px solid #6c8eff;border-radius:10px;font-family:monospace;">
      <div style="color:#8890aa;font-size:11px;margin-bottom:6px;text-transform:uppercase;
        letter-spacing:1px;">Open in your browser</div>
      <a href="{app_url}" target="_blank" rel="noopener"
        style="color:#6c8eff;font-size:20px;font-weight:bold;text-decoration:none;">
        {app_url}
      </a>
      <div style="color:#4caf81;font-size:12px;margin-top:12px;">
        ✓ API key loaded from Colab Secrets &nbsp;|&nbsp;
        ✓ No account required &nbsp;|&nbsp;
        ✓ Encrypted HTTPS
      </div>
    </div>
    '''))

## Cell 6 — Run pipeline stages from Python

You can also do this from the **Run Pipeline** panel in the UI — whichever you prefer.

In [ ]:
import requests

BASE = f'http://localhost:{PORT}/api'

def run_stage(stage, limit=None):
    """Trigger a pipeline stage. Runs in background — check status() for progress."""
    r = requests.post(f'{BASE}/pipeline/run', json={'stage': stage, 'limit': limit})
    r.raise_for_status()
    print(f'Started: {stage}', r.json())

def status():
    """Print current PRISMA flow counts."""
    counts = requests.get(f'{BASE}/pipeline/status').json()['prisma_counts']
    print('\n── Pipeline Status ───────────────────────────')
    for k, v in counts.items():
        marker = ' ⚠' if k == 'needs_human_verification' and v > 0 else ''
        print(f'  {k:42s} {v}{marker}')
    print()

def uncertain():
    """List records needing human verification."""
    data = requests.get(f'{BASE}/records/uncertain/list').json()
    print(f'\nNeeds Human Verification: {data["count"]} records')
    for r in data['records'][:10]:
        print(f'  [{r["record_id"][:8]}] {r["title"][:70]}')
    if data['count'] > 10:
        print(f'  ... and {data["count"]-10} more (see UI)')

def fulltext_needed():
    """List papers where PDFs are needed."""
    data = requests.get(f'{BASE}/records/fulltext-needed/list').json()
    print(f'\nFull Text Needed: {data["count"]} papers')
    for r in data['records'][:10]:
        print(f'  DOI: {r["doi"] or "—":40s}  {r["title"][:50]}')

print('Pipeline helpers loaded.')
print('Usage: run_stage("import") | run_stage("dedup") | run_stage("title_screening", limit=50)')
print('       status() | uncertain() | fulltext_needed()')

In [ ]:
# ── Run the full pipeline ────────────────────────────────
# Uncomment one at a time and check status() between stages.

# run_stage('import')
# status()

# run_stage('dedup')
# status()

# run_stage('title_screening')       # add limit=50 for a test run
# status()
# uncertain()                        # review these in the UI before continuing

# run_stage('fulltext_screening')
# fulltext_needed()                  # upload missing PDFs, then re-run

# run_stage('extraction')
# status()

## Cell 7 — Save progress to Google Drive

**Run this before closing Colab.** Free sessions disconnect after ~12 h of inactivity and all local files are lost.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)

DRIVE_DEST = '/content/drive/MyDrive/SysRev_LLMs_QDA'
os.makedirs(DRIVE_DEST, exist_ok=True)

APP_DIR = '/content/sysrev/systematic-review'

# Pipeline state
state = f'{APP_DIR}/data/pipeline_state.jsonl'
if os.path.exists(state):
    shutil.copy(state, f'{DRIVE_DEST}/pipeline_state.jsonl')
    print('Saved: pipeline_state.jsonl')

# Outputs
out_src = f'{APP_DIR}/data/output'
if os.path.exists(out_src) and os.listdir(out_src):
    shutil.copytree(out_src, f'{DRIVE_DEST}/output', dirs_exist_ok=True)
    print('Saved: output/')

# PDFs (only if you uploaded any)
pdf_src = f'{APP_DIR}/data/pdfs'
if os.path.exists(pdf_src) and os.listdir(pdf_src):
    shutil.copytree(pdf_src, f'{DRIVE_DEST}/pdfs', dirs_exist_ok=True)
    print('Saved: pdfs/')

# Raw search files
raw_src = f'{APP_DIR}/data/raw'
if os.path.exists(raw_src) and os.listdir(raw_src):
    shutil.copytree(raw_src, f'{DRIVE_DEST}/raw', dirs_exist_ok=True)
    print('Saved: raw/')

print(f'\nAll saved to: {DRIVE_DEST}')
!ls -lh {DRIVE_DEST}/

## Cell 8 — Restore session (after Colab resets)

When your session resets, run **Cell 1 → Cell 2 → this cell → Cell 5** to resume exactly where you left off.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)

DRIVE_SRC = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP_DIR   = '/content/sysrev/systematic-review'

if not os.path.exists(DRIVE_SRC):
    print('No saved session found — starting fresh.')
else:
    restored = []

    state = f'{DRIVE_SRC}/pipeline_state.jsonl'
    if os.path.exists(state):
        os.makedirs(f'{APP_DIR}/data', exist_ok=True)
        shutil.copy(state, f'{APP_DIR}/data/pipeline_state.jsonl')
        restored.append('pipeline state')

    for folder in ['output', 'pdfs', 'raw']:
        src = f'{DRIVE_SRC}/{folder}'
        if os.path.exists(src):
            shutil.copytree(src, f'{APP_DIR}/data/{folder}', dirs_exist_ok=True)
            restored.append(folder)

    print(f'Restored: {" | ".join(restored)}')
    print('Now run Cell 5 to restart the server.')

## Cell 9 — Stop the server

In [ ]:
try:
    tunnel.terminate()
    server.terminate()
except Exception:
    pass
os.system('pkill cloudflared 2>/dev/null; true')
print('Server and tunnel stopped.')

---
## Security notes

| Concern | How it is handled |
|---------|------------------|
| API key storage | Colab Secrets — encrypted, never in notebook output or git |
| API key in transit | Written only to a local `.env` inside the Colab VM — never sent over the network |
| Public URL | Cloudflare HTTPS tunnel — no account or token needed; URL is random per session |
| Data persistence | Only what you explicitly save to your own Google Drive |
| Repository | `.env` and `data/` are in `.gitignore` — never committed |

## Quick reference

| Task | How |
|------|-----|
| Change model assignments | UI → Model Config, or edit Cell 2 and rerun |
| Upload search files | Cell 3 (or UI → Full Text Needed for PDFs) |
| Run pipeline | Cell 6, or UI → Run Pipeline |
| Review uncertain records | UI → Human Verification |
| Export evidence table | UI → Export → Download CSV |
| Save before closing | Cell 7 |
| Resume after reset | Cell 1 → 2 → 8 → 5 |
